In [ ]:
!nvidia-smi
!pip install -q -U ultralytics roboflow

Tue Oct 27 15:38:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "YOLO11s_currency_training/currency_security_v8"
)

os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/YOLO11s_currency_training/currency_security_v8


In [ ]:
!pip install -q roboflow ultralytics

from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass("Enter your Roboflow API key: ")

rf = Roboflow(api_key=API_KEY)

project = rf.workspace("as-workspace-q6lmy").project(
    "currency-security-features-b0rck"
)

version = project.version(8)
dataset = version.download("yolov11")

DATASET_PATH = dataset.location

print("Dataset downloaded to:", DATASET_PATH)
print("Dataset YAML:", f"{DATASET_PATH}/data.yaml")

Enter your Roboflow API key: ··········
loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded to: /content/currency-security-features-8
Dataset YAML: /content/currency-security-features-8/data.yaml


In [ ]:
import yaml

with open(f"{DATASET_PATH}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print("Number of classes:", data["nc"])
print("Class names:", data["names"])

assert data["nc"] == 13, (
    "Expected 13 classes. Check the class mapping before training."
)

Number of classes: 13
Class names: ['1000Rs Note', '100Rs Note', '10Rs Note', '20Rs Note', '5000Rs Note', '500Rs Note', '50Rs Note', 'printed_year', 'sbp_seal', 'security_dots', 'security_lines', 'security_thread', 'serial_number']


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=-1,
    patience=20,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    cos_lr=True,
    workers=2,
    seed=42,
    project=PROJECT_DIR,
    name="train",
    exist_ok=True,
    plots=True,
    save=True,
    device=0
)

print("Training complete.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/currency-security-features-8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscri

In [ ]:
from ultralytics import YOLO
import os

BEST_WEIGHTS = os.path.join(
    PROJECT_DIR, "train", "weights", "best.pt"
)

assert os.path.exists(BEST_WEIGHTS), "best.pt was not created."

best_model = YOLO(BEST_WEIGHTS)

metrics = best_model.val(
    data=f"{DATASET_PATH}/data.yaml",
    imgsz=640,
    split="test",
    plots=True
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Per-class mAP50:", metrics.box.maps)

Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,417,831 parameters, 0 gradients, 21.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 15.8±11.0 MB/s, size: 49.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/currency-security-features-8/test/labels... 388 images, 0 backgrounds, 40 corrupt: 100% ━━━━━━━━━━━━ 388/388 300.0it/s 1.3s
val: /content/currency-security-features-8/test/images/20260812_162254_jpg.rf.af26487a331335f31d38ff763342b031.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/currency-security-features-8/test/images/20260812_162827_jpg.rf.54f5b6567cc24c3cb4570d9f0bdd23e3.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/currency-security-features-8/test/images/20260812_163141_jpg.rf.8a5